In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # Load environment variables from .env file
openai_client = OpenAI()  # Initialize the OpenAI client

In [18]:
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)  # Build the index from the loaded documents

In [3]:
documents

[{'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful 

In [20]:
from rag_helper import RAGBase


In [5]:
# Importing our sentence transformer model
from sentence_transformers import SentenceTransformer

# Creating a model object
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])

In [26]:
texts = []

for doc in documents:
    text =doc['question'] + " " + doc['answer']
    texts.append(text)

In [27]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [28]:
import numpy as np
X = np.array(vectors)

In [29]:
vindex.fit(X, documents)

In [30]:
query = "I just found out about the program, can I join?"

Text search performs well but we can still avoid many issues if we use vector search.

In [31]:
# we add the embedder to our RAG class then we use it to vectorize to the query and search the index for the most relevant documents inside the search method.
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs): # stores an embedder instance (plus any RAGBase kwargs)
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

 The embedder is passed as an argument to the RAGVector class and is used to encode the query into a vector representation. The search method then uses this vector to perform a similarity search in the index, returning the most relevant documents based on the query.

In [32]:
# Creating an instance of RAGVector with the embedder, index, and llm_client
vector_assistant = RAGVector(
    embedder = model,
    index=vindex,
    llm_client=openai_client,
)

In [33]:
query = "I just found out about the program, can I join?"
vector_results = vector_assistant.rag(query)

In [34]:
vector_results


'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [40]:
vs_index.fit(vectors, documents)

In [47]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query) # Always encode the query before vector searching 

results = vs_index.search(query_vector, filter_dict={"course": "llm-zoomcamp"}, num_results=5)

In [48]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [49]:
vs_index.close()